# Ethan's Canvas AI Tutor

An AI agent connected to my Canvas LMS that can:
- List all my active courses
- Read my syllabus and course modules
- Check upcoming assignments and quizzes
- Explain assignments step-by-step

> **Built with LangGraph + LangChain + Gemini**  
> **Observability powered by LangSmith** — watch [LangSmith 101 by James Briggs](https://www.youtube.com/watch?v=Iyc80hY2yYk) for a full walkthrough of everything this notebook uses.

Your `.env` file must contain:
```
GEMINI_API_KEY=...
LANGSMITH_API_KEY=...
CANVAS_API_TOKEN=...
CANVAS_BASE_URL=https://byui.instructure.com/
```


## Step 1 — Install Packages

In [1]:
# Upgrade LangChain packages safely
# pandas is pinned to 2.2.2 — Colab's built-in tools require exactly this version
!pip install -q -U langchain langgraph langchain-google-genai python-dotenv requests langsmith
!pip install -q "pandas==2.2.2"

print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.5/339.5 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
✅ All packages installed!


## Step 2 — Load Keys from Google Drive

In [2]:
import os
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

ENV_PATH = "/content/drive/MyDrive/ai_agents/.env"
load_dotenv(ENV_PATH, override=True)

for key in ["GEMINI_API_KEY", "LANGSMITH_API_KEY", "CANVAS_API_TOKEN", "CANVAS_BASE_URL"]:
    os.environ.pop(key, None)
load_dotenv(ENV_PATH, override=True)

GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
CANVAS_TOKEN      = os.getenv("CANVAS_API_TOKEN")
CANVAS_BASE_URL   = os.getenv("CANVAS_BASE_URL", "https://byui.instructure.com")

assert GEMINI_API_KEY,    "GEMINI_API_KEY not found in .env"
assert LANGSMITH_API_KEY, "LANGSMITH_API_KEY not found in .env"
assert CANVAS_TOKEN,      "CANVAS_API_TOKEN not found in .env"

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY # Fix Jame's Problem here

# LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]   = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"]    = LANGSMITH_API_KEY
os.environ["LANGCHAIN_PROJECT"]    = "canvas-tutor"

print("✅ Keys loaded!")
print(f"Canvas URL:          {CANVAS_BASE_URL}")
print(f"Gemini key    ends in: ...{GEMINI_API_KEY[-4:]}")
print(f"LangSmith key ends in: ...{LANGSMITH_API_KEY[-4:]}")
print(f"Canvas token  ends in: ...{CANVAS_TOKEN[-4:]}")
print("🔍 LangSmith tracing is ON  →  visit smith.langchain.com → canvas-tutor")

Mounted at /content/drive
✅ Keys loaded!
Canvas URL:          https://byui.instructure.com/
Gemini key    ends in: ...mUtM
LangSmith key ends in: ...b347
Canvas token  ends in: ...Pwvh
🔍 LangSmith tracing is ON  →  visit smith.langchain.com → canvas-tutor


## Step 3 — Imports

In [3]:
import re
import requests
from datetime import datetime

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent   # moved to langgraph.prebuilt in LangChain v0.2+
from langsmith import traceable, Client             # traceable wraps any function as a traced span
from IPython.display import display, Markdown

print("✅ Imports ready!")

✅ Imports ready!


## Step 4 — Canvas API Helper Functions with `@traceable`

These functions call the Canvas REST API directly.
Each one is decorated with `@traceable` so that every API call shows up as its own
named span inside LangSmith traces — you can see exactly which endpoint was hit,
how long it took, and what it returned.

This is the pattern James Briggs demonstrates in LangSmith 101: use `@traceable` to
add visibility to *any* code, not just LangChain objects.


In [4]:
HEADERS = {"Authorization": f"Bearer {CANVAS_TOKEN}"}
API_URL  = CANVAS_BASE_URL.rstrip("/") + "/api/v1"


def canvas_get(path: str, params: dict = None) -> dict:
    """Make a GET request to the Canvas API. Returns error dict on 404 instead of raising."""
    resp = requests.get(f"{API_URL}/{path}", headers=HEADERS, params=params or {})
    if resp.status_code == 404:
        return {"error": f"Not found: /{path} — item may be deleted or unpublished"}
    resp.raise_for_status()
    return resp.json()


def safe_int(value) -> int:
    """Safely convert Canvas IDs to int — they can come back as floats from the LLM."""
    return int(float(value))


@traceable(name="Canvas API: list_courses")
def _get_all_courses_raw() -> list:
    """Paginate through all active courses. Traced so latency is visible in LangSmith."""
    courses, page = [], 1
    while True:
        batch = canvas_get("courses", {"enrollment_state": "active", "per_page": 100, "page": page})
        if not batch or isinstance(batch, dict):
            break
        courses.extend(batch)
        page += 1
    return courses


@traceable(name="Canvas API: list_assignments")
def _fetch_assignments_for_course(course_id, course_name) -> list:
    """Fetch future assignments for one course. Traced individually per course."""
    assignments = canvas_get(
        f"courses/{course_id}/assignments",
        {"bucket": "future", "order_by": "due_at", "per_page": 100}
    )
    if isinstance(assignments, dict) and "error" in assignments:
        return []
    for a in assignments:
        a["course_name"] = course_name
    return assignments


@traceable(name="Canvas API: get_upcoming_assignments")
def _get_upcoming_assignments_raw(days_ahead: int = 30) -> list:
    """Fetch upcoming assignments across all courses, sorted by due date."""
    courses = _get_all_courses_raw()
    all_assignments = []
    for course in courses:
        try:
            all_assignments.extend(
                _fetch_assignments_for_course(course["id"], course.get("name", "Unknown"))
            )
        except Exception:
            pass
    return sorted(all_assignments, key=lambda a: a.get("due_at") or "9999")


@traceable(name="Canvas API: get_syllabus")
def _fetch_syllabus(course_id: int) -> str:
    """Fetch syllabus body or fall back to module list."""
    data = canvas_get(f"courses/{safe_int(course_id)}", {"include[]": "syllabus_body"})
    if "error" in data:
        return data["error"]
    html = data.get("syllabus_body") or ""
    text = re.sub(r"<[^>]+>", " ", html)
    text = re.sub(r"\s+", " ", text).strip()
    if text:
        return text[:3000]
    modules = canvas_get(f"courses/{safe_int(course_id)}/modules",
                         {"include[]": "items", "per_page": 100})
    if modules and not isinstance(modules, dict):
        lines = ["Course outline from modules:"] + [f"  Module: {m.get('name','?')}" for m in modules]
        return "\n".join(lines)
    return "(No syllabus or modules available)"


@traceable(name="Canvas API: get_assignment_detail")
def _fetch_assignment_detail(course_id: int, assignment_id: int) -> dict:
    """Fetch full assignment data. Returns error dict on 404."""
    return canvas_get(f"courses/{safe_int(course_id)}/assignments/{safe_int(assignment_id)}")


print("✅ Canvas helper functions ready with @traceable spans")

✅ Canvas helper functions ready with @traceable spans


## Step 5 — Define LangChain Tools

Each `@tool` wraps a `@traceable` helper. In LangSmith you'll see:
```
▼ tools
  ▼ get_upcoming_assignments              ← the @tool node
    ▼ Canvas API: get_upcoming_assignments ← @traceable span
      ▼ Canvas API: list_courses           ← nested @traceable span
      ▼ Canvas API: list_assignments       ← one span per course
```
This full call tree is the main benefit of combining `@tool` with `@traceable`.


In [5]:
@tool
def get_all_courses(placeholder: str = "") -> str:
    """List every active course the student is currently enrolled in.
    Returns course IDs, names, and course codes.
    Call this first before using any other tool that needs a course_id.
    """
    courses = _get_all_courses_raw()
    if not courses:
        return "No active courses found."
    lines = [
        f"ID {c['id']} | {c.get('course_code',''):>12} | {c.get('name','Unnamed')}"
        for c in courses
    ]
    return f"Found {len(courses)} active course(s):\n" + "\n".join(lines)


@tool
def get_upcoming_assignments(days_ahead: int = 30) -> str:
    """Get all upcoming assignments across ALL courses, sorted by due date.
    Use this when the student asks about upcoming work, deadlines, or what is due soon.
    days_ahead controls how far into the future to look (default 30 days).
    """
    assignments = _get_upcoming_assignments_raw(days_ahead)
    if not assignments:
        return f"No upcoming assignments in the next {days_ahead} days."
    lines = []
    for a in assignments[:20]:
        due = a.get("due_at", "")[:10] if a.get("due_at") else "No due date"
        lines.append(
            f"[{due}] {a.get('course_name','')} — {a.get('name','')} "
            f"(ID:{a.get('id','')} | {a.get('points_possible',0)} pts)"
        )
    return "\n".join(lines)


@tool
def get_syllabus(course_id: int) -> str:
    """Get the syllabus for a specific course as plain text.
    Falls back to listing module names if the syllabus body is empty.
    Use this to understand grading policies, course expectations, or topics covered.
    """
    return _fetch_syllabus(course_id)


@tool
def get_assignment_detail(course_id: int, assignment_id: int) -> str:
    """Get the full description and requirements for one specific assignment.
    Use this when the student wants to understand exactly what an assignment asks for.
    Requires both course_id and assignment_id — get these from get_upcoming_assignments.
    """
    data = _fetch_assignment_detail(course_id, assignment_id)
    if "error" in data:
        return f"Could not retrieve assignment: {data['error']}. Try a different assignment."
    html = data.get("description") or ""
    desc = re.sub(r"<[^>]+>", " ", html)
    desc = re.sub(r"\s+", " ", desc).strip()
    return (
        f"Assignment: {data.get('name','')}\n"
        f"Points: {data.get('points_possible', 0)}\n"
        f"Due: {data.get('due_at','Not set')}\n"
        f"Description:\n{desc[:2000] if desc else '(No description provided)'}"
    )


@tool
def get_quizzes(course_id: int) -> str:
    """List all quizzes for a course with their due dates and point values.
    Use this when the student asks about quizzes or tests in a specific course.
    """
    quizzes = canvas_get(f"courses/{safe_int(course_id)}/quizzes", {"per_page": 100})
    if isinstance(quizzes, dict) and "error" in quizzes:
        return quizzes["error"]
    if not quizzes:
        return "No quizzes found for this course."
    lines = [
        f"• {q.get('title','')} | {q.get('points_possible',0)} pts | Due: {q.get('due_at','No due date')}"
        for q in quizzes
    ]
    return "\n".join(lines)


@tool
def get_modules(course_id: int) -> str:
    """Get the learning modules and their items for a course.
    Use this to understand the course structure, readings, and available content.
    """
    modules = canvas_get(f"courses/{safe_int(course_id)}/modules",
                         {"include[]": "items", "per_page": 100})
    if isinstance(modules, dict) and "error" in modules:
        return modules["error"]
    if not modules:
        return "No modules found for this course."
    lines = []
    for m in modules:
        lines.append(f"\nModule: {m.get('name','')}")
        for item in m.get("items", []):
            lines.append(f"  - [{item.get('type','')}] {item.get('title','')}")
    return "\n".join(lines)


TOOLS = [get_all_courses, get_upcoming_assignments, get_syllabus,
         get_assignment_detail, get_quizzes, get_modules]
print(f"✅ {len(TOOLS)} tools registered: {[t.name for t in TOOLS]}")
print("🔍 Each tool's @traceable helpers will appear as nested spans in LangSmith")

✅ 6 tools registered: ['get_all_courses', 'get_upcoming_assignments', 'get_syllabus', 'get_assignment_detail', 'get_quizzes', 'get_modules']
🔍 Each tool's @traceable helpers will appear as nested spans in LangSmith


## Step 6 — Build the Tutor Agent

In [6]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)

today_str = datetime.now().strftime("%A, %B %d, %Y")

SYSTEM_PROMPT = (
    f"You are a knowledgeable and encouraging AI tutor connected to the student's Canvas LMS. "
    f"Today is {today_str}. "
    "Always fetch real Canvas data before answering — never guess course IDs or assignment names. "
    "For questions about upcoming work, call get_upcoming_assignments first. "
    "Read the syllabus to understand grading policies and course expectations. "
    "Explain assignments clearly: what is needed, how to approach it, what the rubric says. "
    "Help the student plan and prioritize their work. "
    "Be encouraging, clear, and specific."
)

agent = create_react_agent(
    model=llm,
    tools=TOOLS,
    prompt=SYSTEM_PROMPT,
)

print("✅ Canvas tutor agent ready!")
print("🔍 Every run → smith.langchain.com → canvas-tutor project")

✅ Canvas tutor agent ready!
🔍 Every run → smith.langchain.com → canvas-tutor project


/tmp/ipython-input-267/146753306.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Step 7 — Ask the Tutor

Run any cell. In LangSmith → **canvas-tutor** you'll now see the full nested trace:
each `@tool` call expands to show every `@traceable` Canvas API span inside it,
with individual latency for each HTTP request.


In [7]:
result = agent.invoke({"messages": [{"role": "user", "content": "What courses am I enrolled in right now?"}]})
display(Markdown(f"**Tutor:** {result['messages'][-1].content}"))

**Tutor:** OK. You are currently enrolled in: BUS 301 (Adv Writing in Pro Contexts), DS 460 (Big Data Programming), CSE Majors, Math Tutor, Mathematics Majors, CSE 490R (Special Topics), and MATH 488 (Stat & Data Sci Consulting). Is there anything I can help you with regarding these courses? For example, I can list upcoming assignments, get the syllabus, or get details about a specific assignment.

In [8]:
result = agent.invoke({"messages": [{"role": "user", "content": "What assignments do I have due in the next two weeks? Sort them by due date."}]})
display(Markdown(f"**Tutor:** {result['messages'][-1].content}"))

**Tutor:** Okay, here's a breakdown of your assignments due in the next two weeks, starting with the earliest:

**Tomorrow, February 27th:**

*   **Big Data Programming:**
    *   Pyspark Data Types Written Challenge (4.0 pts)
    *   Written Challenge 3 - PySpark Data Types (0.0 pts)

**Friday, February 28th:**

*   **Special Topics:** Sprint Presentation (100.0 pts)

**Sunday, March 1st:**

*   **Big Data Programming:** Week 8 - Stand Up Report (1.0 pts)

**Tuesday, March 3rd:**

*   **Adv Writing in Pro Contexts:**
    *   Assignment: Writing with AI --Proposing a Four-Day Workweek (60.0 pts)
    *   Week 07: Reading Accountability Quiz (10.0 pts)
*   **Big Data Programming:** Pyspark Coding Challenge (4.0 pts)

**Thursday, March 5th:**

*   **Big Data Programming:** API Data Extraction and Loading Challenge (4.0 pts)

**Sunday, March 8th:**

*   **Big Data Programming:** Week 9 - Stand Up Report (1.0 pts)

**Sunday, March 15th:**

*   **Big Data Programming:** Week 10 - Stand Up Report (1.0 pts)

It looks like you have a busy couple of weeks! Would you like me to get the details for any of these assignments, or perhaps help you prioritize them? For example, I could grab the full description of the "Writing with AI" assignment in Adv Writing in Pro Contexts. Let me know what would be most helpful.

In [9]:
result = agent.invoke({"messages": [{"role": "user", "content": "Find my most urgent assignment. Read its full description and explain step-by-step what I need to do."}]})
display(Markdown(f"**Tutor:** {result['messages'][-1].content}"))

**Tutor:** Okay, I've got the details for your most urgent assignment:

**Course:** Big Data Programming
**Assignment:** Pyspark Data Types Written Challenge
**Due Date:** Tomorrow, February 27th by 6:59 AM
**Points:** 4

**Here's what you need to do:**

1.  **Prepare:** Grab a single sheet of paper and a writing utensil.
2.  **Answer Questions:** Answer the questions that will be presented in class on the sheet of paper.
3.  **Include Details:** Write your full name and the time of your class at the top of the paper.
4.  **Submit:** Bring your completed sheet of paper to class.

This assignment seems straightforward. Make sure you attend class and are ready to answer the questions on Pyspark Data Types. Good luck!

## Step 8 — LangSmith Evaluation Dataset

Same pattern as the Wellness Agent — create a dataset of expected inputs and a tool-routing
evaluator, then run `evaluate()` to get a scored experiment in LangSmith.


In [10]:
from langsmith.evaluation import evaluate

client = Client()

DATASET_NAME = "canvas-tutor-eval"

existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Input/output pairs for evaluating the Canvas tutor agent"
    )
    client.create_examples(
        inputs=[
            {"input": "What courses am I enrolled in?"},
            {"input": "What assignments do I have due this week?"},
            {"input": "What is on the syllabus for my AI class?"},
            {"input": "Show me the modules for course 12345"},
            {"input": "What quizzes are coming up?"},
        ],
        outputs=[
            {"expected_tool": "get_all_courses"},
            {"expected_tool": "get_upcoming_assignments"},
            {"expected_tool": "get_syllabus"},
            {"expected_tool": "get_modules"},
            {"expected_tool": "get_quizzes"},
        ],
        dataset_id=dataset.id,
    )
    print(f"✅ Dataset '{DATASET_NAME}' created with 5 examples")
else:
    print(f"✅ Dataset '{DATASET_NAME}' already exists — reusing it")

✅ Dataset 'canvas-tutor-eval' already exists — reusing it


In [11]:
def correct_tool_called(inputs, outputs, reference_outputs):
    """Returns 1 if the agent called the expected tool, 0 otherwise."""
    expected = reference_outputs.get("expected_tool", "")
    messages = outputs.get("messages", [])
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for call in msg.tool_calls:
                if call.get("name") == expected:
                    return {"score": 1, "comment": f"Called '{expected}' ✅"}
    return {"score": 0, "comment": f"Expected '{expected}' but not found ❌"}


def run_agent(inputs):
    return agent.invoke({"messages": [{"role": "user", "content": inputs["input"]}]})


results = evaluate(
    run_agent,
    data=DATASET_NAME,
    evaluators=[correct_tool_called],
    experiment_prefix="canvas-tool-selection",
)

print("\n✅ Evaluation complete!")
print("View results at smith.langchain.com → canvas-tutor → Experiments tab")

View the evaluation results for experiment: 'canvas-tool-selection-66ee5262' at:
https://smith.langchain.com/o/77e8dde6-572b-4549-95f9-e2954451f649/datasets/0a291ecd-3405-45ad-adb0-f091f5f26e35/compare?selectedSessions=09ccb15c-bc22-42c6-b8f1-c13fc06ac35e




0it [00:00, ?it/s]


✅ Evaluation complete!
View results at smith.langchain.com → canvas-tutor → Experiments tab


## Step 9 — Interactive Chat

In [ ]:
print("Canvas AI Tutor — type your question ('quit' to stop)\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye! Good luck with your studies! 🎓")
        break

    result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    display(Markdown(f"**Tutor:** {result['messages'][-1].content}"))
    print()

Canvas AI Tutor — type your question ('quit' to stop)



**Tutor:** I cannot determine which course is the "hardest" for you because difficulty is subjective and depends on your individual strengths, weaknesses, and interests. I can access course syllabi and assignment details to help you understand the workload and expectations for each course. Would you like me to get the syllabi for your courses, or list upcoming assignments so you can compare the workload?